**Tutorial series (2 of 4):**  [1 · Metagenomics](metagenomics.ipynb) · [2 · Build experimental DB](build_experimental_db.ipynb) · [3 · Parameter tuning](parameter_tuning.ipynb) · [4 · Modeling](modeling.ipynb)

# Building an experimental dataset for calibration — from a paper to `Experiment` objects

*A companion to the [metagenomics notebook](metagenomics.ipynb).* There, we turned
sequencing reads into a **day-0 microbial COD allocation**. Here we take the
**measured performance data** from the same study — the volatile fatty acids
(VFAs) and methane reported by Ding *et al.* — and turn them into ADToolbox
`Experiment` objects that a parameter-tuning notebook can train against.

The point of this notebook is to teach the two things that trip people up when
they try to fit a model to published data:

1. **Units.** A published table of "acetic acid, mg/L" is *not* what the model
   sees. The e-ADM model lives in **chemical oxygen demand (COD), gCOD/L**. We'll
   convert measurements to COD **using ADToolbox's own compound database**, the
   same way the toolkit does internally — so your data and your model speak the
   same language.
2. **Assembling a valid initial state.** An `Experiment` is more than a table of
   numbers: it needs a starting chemical state, the microbial seed (from the
   metagenomics notebook), the feed, and the reactor operating mode.

> **Study.** Ding *et al.* (2022), *Influence of Inoculum Type on VFA and Methane
> Production in Short-Term Anaerobic Food Waste Digestion Tests*, ACS Sustain.
> Chem. Eng. 10(51):17071–17080 — DOI `10.1021/acssuschemeng.2c04080`. Food waste
> digested with three inocula (**AS**, **TWAS**, **TWAS+AS**), two replicates,
> sampled on days **0, 1, 5, 11, 18, 24**.


## 1. Setup & prerequisites

You need two things this notebook builds on:

- **The ADToolbox databases** (the same `database_dir` used in the metagenomics
  notebook). We use the SEED compound database inside it to compute COD.
- **The day-0 COD allocation** produced by the metagenomics notebook (per-sample
  `cod_profile.csv`). That gives us the microbial seed for each bottle.

Edit the three paths below to match your machine.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from adtoolbox import core, configs, utils

REPO = Path.cwd().parent if Path.cwd().name == "Examples" else Path.cwd()

DATABASE_DIR = REPO / "database"                          # ADToolbox databases (as in the metagenomics notebook)
COD_DIR      = REPO / "tutorial_output" / "ding_day0"     # metagenomics-notebook output (per-sample cod_profile.csv)
RAW          = REPO / "Examples" / "Studies" / "ding_raw" # raw supplemental VFA/methane tables (ship with the repo)
MODEL_DB     = REPO / "reference_data" / "models.json"
DOI = "https://doi.org/10.1021/acssuschemeng.2c04080"

CONDITIONS = ["FW+TWAS", "FW+AS", "FW+TWAS+AS"]
REPS = [1, 2]
TIMES = [0, 1, 5, 11, 18, 24]                             # sampling days
VARIABLES = ["S_ac", "S_pro", "S_bu", "S_va", "S_cap"]    # the measured VFAs, as model species names
print("database dir:", DATABASE_DIR)

## 2. What the paper actually reports (and in what units)

Read the supplement carefully before touching a model. For this study:

| quantity | reported as | what it means |
| --- | --- | --- |
| individual VFAs | **mg/L** | soluble concentration in the reactor liquid |
| total VFA | g/L | sum of the individual acids (peaks at **6.62 g/L**) |
| methane | **cumulative mL** | gas produced up to that day, from GC (min **296 mL**) |

Two traps live in that table: the VFAs are a **mass** concentration (needs a COD
conversion), and methane is a **cumulative gas volume** (not a concentration at
all). We handle each below.

## 3. COD — the model's currency, computed with ADToolbox

The model tracks everything as **oxygen demand**. To convert `X mg/L` of an acid
to `gCOD/L`, we need that acid's COD per gram. Rather than hard-code textbook
numbers, we ask ADToolbox's **SEED compound database** — the exact object the
toolkit uses to build its own COD tables — via `SeedDB.instantiate_metabs(...)`
and `Metabolite.cod_calc()`. Each VFA is identified by its SEED compound id.

In [ ]:
seed_db = core.SeedDB(configs.Database(database_dir=str(DATABASE_DIR)))

SEED_ID = {"S_ac": "cpd00029", "S_pro": "cpd00141", "S_bu": "cpd00211",
           "S_va": "cpd00597", "S_cap": "cpd01113"}          # acetate ... caproate
CH4_ID = "cpd01024"

# cod_calc returns gCOD per mol-equivalent; /1000 gives the factor for mg/L -> gCOD/L
COD_FACTOR = {sp: seed_db.instantiate_metabs(cid).cod_calc(add_h=1) / 1000
              for sp, cid in SEED_ID.items()}
CH4_GCOD_PER_ML = seed_db.instantiate_metabs(CH4_ID).cod_calc() / 1000 * (1/22.414) * 1000
# ^ gCOD per mg CH4 -> per mL at STP:  (gCOD/mg)*(mg? ) — see note below; we recompute cleanly:
CH4_GCOD_PER_ML = 64.0 / 22414.0    # 1 mol CH4 = 64 gCOD occupies 22.414 L at STP

print("VFA COD factors (mg/L -> gCOD/L), from the SEED database:")
for sp in VARIABLES:
    print(f"  {sp:6} {SEED_ID[sp]}  {COD_FACTOR[sp]:.6f}")
print(f"methane: 1 mL CH4 -> {CH4_GCOD_PER_ML:.5f} gCOD (STP)")

## 4. From raw VFA tables (mg/L) to model VFAs (gCOD/L)

The supplement lists **iso** and **normal** isomers of butyric, valeric, and
caproic acids separately. The model has one lumped species for each, so we **sum
the isomers**, then multiply by the COD factor. The result is a `(time × VFA)`
matrix in gCOD/L — exactly what an `Experiment` stores as its fitting `data`.

In [ ]:
def read_conc(fname):
    return pd.read_csv(RAW / "concentrations" / fname, sep="\t", index_col=0)

def vfa_mg_per_L(condition, rep):
    row = f"{condition}_rep_{rep}"
    return {
        "S_ac":  read_conc("Acetic.txt").loc[row],
        "S_pro": read_conc("Propionic.txt").loc[row],
        "S_bu":  read_conc("n_butyric.txt").loc[row] + read_conc("i_butyric.txt").loc[row],
        "S_va":  read_conc("n_valeric.txt").loc[row] + read_conc("i_valeric.txt").loc[row],
        "S_cap": read_conc("n-caproic.txt").loc[row] + read_conc("i_caproic.txt").loc[row],
    }

def vfa_gcod(condition, rep):
    mg = vfa_mg_per_L(condition, rep)
    return np.column_stack([mg[v].to_numpy(float) * COD_FACTOR[v] for v in VARIABLES])

demo = pd.DataFrame(vfa_gcod("FW+TWAS", 1), index=TIMES, columns=VARIABLES)
print("FW+TWAS rep 1 — VFAs in gCOD/L:")
print(demo.round(3))
# sanity: does the total (in g/L, mass) reproduce the paper's ~6.6 g/L peak?
tot = max(sum(vfa_mg_per_L("FW+TWAS", r).values()).max() for r in REPS) / 1000
print(f"\npeak total VFA (mass) for FW+TWAS: {tot:.2f} g/L   (paper: 6.62 g/L)")

## 5. Methane — the units trap

Methane is a **cumulative gas volume (mL)**, so two conversions stand between it
and the model: mL → gCOD (at STP, `64 gCOD / 22.414 L`), and gCOD → concentration
(divide by the bottle's **working liquid volume**). We *don't have the bottle
volume from the open abstract*, so treat `V_LIQ` as a knob to set from the paper's
methods. This is why we do **not** fit methane until the volume is pinned — a
wrong volume rescales the entire target.

In [ ]:
methane = pd.read_csv(RAW / "methane_values.csv")
methane.columns = ["condition"] + [int(c) for c in methane.columns[1:]]
methane = methane.set_index("condition")

V_LIQ_L = 0.1     # <-- bottle working volume in LITRES; SET FROM THE PAPER METHODS
methane_gcod_per_L = methane * CH4_GCOD_PER_ML / V_LIQ_L
print("cumulative methane (mL):"); print(methane)
print(f"\n-> gCOD/L with V_liq={V_LIQ_L} L (PLACEHOLDER):"); print(methane_gcod_per_L.round(2))

## 6. The microbial seed — handoff from the metagenomics notebook

The bottles were inoculated with different sludges, so each starts with a
different microbial community. The **metagenomics notebook** already quantified
that: its per-sample `cod_profile.csv` gives the **relative COD share of each
functional group** at day 0. We scale those shares by the sludge loading of each
bottle (from the study) to get absolute biomass, in gCOD/L.

In [ ]:
LOADING = {"FW+TWAS": 3.25, "FW+AS": 0.75, "FW+TWAS+AS": 4.0}   # gCOD sludge added per bottle (study SI)

def biomass_ic(condition, rep):
    """Day-0 X_* biomass (gCOD/L) = metagenomics COD shares x sludge loading."""
    d = COD_DIR / f"{condition.replace('+','_')}_rep_{rep}" / "cod_profile.csv"
    prof = pd.read_csv(d).set_index("group")["value"]
    prof = prof / prof.sum()                                   # normalise to shares
    return {g: float(prof[g]) * LOADING[condition] for g in prof.index}

bm = biomass_ic("FW+TWAS", 1)
print("FW+TWAS rep 1 — day-0 biomass (gCOD/L), top groups:")
print(pd.Series(bm).sort_values(ascending=False).head(6).round(3).to_string())

## 7. Assembling a training-ready `Experiment`

Now we put the pieces together. A good `Experiment` for calibration has:

- **`variables` / `data`** — the measured VFAs (names, gCOD/L) the optimizer fits.
- **`initial_concentrations`** — the full starting state: VFAs seeded from the
  measured **t = 0** (not zero!), the microbial seed from §6, the feed's
  particulate/soluble COD (`TSS`/`TDS` from the 18 gCOD/L feed), and the acid–base
  split of each VFA at the starting pH.
- **`base_parameters`** — batch operation: no flow (`q_in = 0`) and small closed
  volumes.
- **`feed`, `constants`, `reference`** — provenance and the pH we hold fixed.

In [ ]:
p = utils.load_model_json(str(MODEL_DB), "e_adm")
Ka = p["model_parameters"]
feed = core.Feed(name="Foodwaste", carbohydrates=50.1, lipids=21.5, proteins=20.5,
                 tss=30, si=47, xi=22.5)                       # FW characterisation (paper + refs)
TOTAL_FEED_COD, pH0 = 18.0, 6.5                                # gCOD/L (paper), starting pH
TSS = TOTAL_FEED_COD * feed.tss / 100; TDS = TOTAL_FEED_COD - TSS
S_H = 10 ** (-pH0)

def build_experiment(condition, rep):
    data = vfa_gcod(condition, rep)                            # (time, VFA) gCOD/L
    ic = {v: float(data[0, j]) for j, v in enumerate(VARIABLES)}   # VFAs at t=0
    ic.update(biomass_ic(condition, rep))                     # microbial seed
    # acid-base: the fraction of each VFA that is dissociated at the starting pH
    for v, ka in [("S_va","K_a_va"),("S_bu","K_a_bu"),("S_pro","K_a_pro"),
                  ("S_cap","K_a_cap"),("S_ac","K_a_ac")]:
        ic[f"{v}_ion"] = Ka[ka] / (Ka[ka] + S_H) * ic[v]
    ic.update({"S_su": 0.0, "S_aa": 0.0, "S_fa": 0.0,
               "TSS": TSS, "TDS": TDS, "S_H_ion": S_H})
    return core.Experiment(
        name=f"{condition.replace('+','_')}_rep_{rep}",
        time=list(TIMES), variables=list(VARIABLES), data=data.T.tolist(),
        feed=feed, initial_concentrations=ic,
        base_parameters={"q_in": 0, "V_liq": 0.0001, "V_gas": 0.00007},
        constants=["S_H_ion"], reference=DOI, model_name="e_adm")

experiments = [build_experiment(c, r) for c in CONDITIONS for r in REPS]
e = experiments[0]
print(f"built {len(experiments)} experiments")
print(f"{e.name}: variables={e.variables}")
print("  t0 VFAs:", {v: round(e.initial_concentrations[v], 3) for v in VARIABLES})

## 8. Sanity checks

A rebuilt dataset should (a) reproduce the paper's headline VFA total, and (b)
agree with any prior version of the data. Here we confirm the VFAs match the
database shipped with ADToolbox — and note the one interface difference: the old
records stored `variables` as **species indices**, whereas we store **names**,
which the current tuning interface expects.

In [ ]:
EX = REPO / "Examples"
db = core.Database(configs.Database(feed_db=EX / "feed_db.tsv",
    studies_local={"metagenomics_studies": EX / "Studies" / "metagenomics_studies.tsv",
                   "experimental_data_db": EX / "Studies" / "experimental_data_references.json"}))
old = {o.name: o for o in db.get_experiment_from_experiments_db("reference", DOI)}
o = old["fw_twas_rep_1"]; new = build_experiment("FW+TWAS", 1)
print("shipped-DB variables:", list(o.variables), " (species indices)")
print("our variables      :", list(new.variables), " (names)")
print("VFA values agree (atol 0.02)?",
      np.allclose(np.asarray(o.data, float), np.asarray(new.data, float), atol=0.02))

## 9. Save — ready for parameter tuning

We serialise the experiments to one JSON in the format a tuning notebook can load
(name, time, variable **names**, data, feed, full initial state, batch
parameters). Point the parameter-tuning notebook's study file at this to train on
the corrected, unit-checked dataset.

In [ ]:
out = REPO / "Examples" / "Studies" / "ding_experiments.json"
payload = {}
for (c, r), e in zip([(c, r) for c in CONDITIONS for r in REPS], experiments):
    payload[e.name] = {
        "name": e.name, "condition": c, "reference": e.reference, "model_name": e.model_name,
        "time": e.time, "variables": e.variables, "data": np.asarray(e.data, float).T.tolist(),   # (variable, time): core.Experiment transposes back
        "feed": {"name": feed.name, "carbohydrates": feed.carbohydrates, "lipids": feed.lipids,
                 "proteins": feed.proteins, "tss": feed.tss, "si": feed.si, "xi": feed.xi},
        "initial_concentrations": e.initial_concentrations,
        "base_parameters": e.base_parameters, "constants": e.constants,
    }
out.write_text(json.dumps(payload, indent=1))
print("wrote", out, "(", len(payload), "experiments )")

## What you learned

- **Convert measurements to COD with the toolkit's own SEED database** — never
  eyeball unit factors; `SeedDB` + `cod_calc()` keeps data and model consistent.
- **Sum reported isomers** into the model's lumped VFA species.
- **Seed the initial state properly**: measured t=0 VFAs, the metagenomics-derived
  microbial community, the feed's COD split, and the acid–base speciation at the
  starting pH.
- **Methane is cumulative gas volume** — convert at STP *and* by the reactor
  volume, and don't fit it until that volume is known.

The saved `ding_experiments.json` is the clean, calibration-ready product of this
notebook — the input to parameter tuning.
